In [ ]:
# Q0 - SETUP

import re, random, hashlib, string
import pandas as pd
import nltk
import matplotlib.pyplot as plt

from nltk.tokenize import word_tokenize, sent_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet, stopwords
from collections import Counter
from nltk.util import bigrams

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg)

# --- Dataset Generation ---
def generate_dataset(roll_no):
    rng = random.Random(int(roll_no))
    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }
    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)
    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

STOP_WORDS = set(stopwords.words("english"))

def check_punct(tok):
    return all(c in string.punctuation for c in tok)

def analyze_ticket(text):
    sents = sent_tokenize(text)
    all_tokens = [w.lower() for s in sents for w in word_tokenize(s)]
    punctuation_tokens = [w for w in all_tokens if check_punct(w)]
    stopword_tokens = [w for w in all_tokens if w in STOP_WORDS]
    content_tokens = [w for w in all_tokens if not check_punct(w) and w not in STOP_WORDS]
    return {
        "sentences": sents,
        "tokens": all_tokens,
        "punct": punctuation_tokens,
        "stops": stopword_tokens,
        "final": content_tokens
    }

dataset, FOCUS, FINGERPRINT = generate_dataset(1)
tickets = dataset['Ticket_Text'].tolist()

TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

In [ ]:
# Q1

table_data = []
for idx, t in enumerate(tickets, start=1):
    result = analyze_ticket(t)
    table_data.append({
        "Ticket": idx,
        "Sentences": len(result["sentences"]),
        "Tokens": len(result["tokens"]),
        "Punctuation": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique Tokens": len(set(result["tokens"])),
        "Final Tokens": len(result["final"])
    })

summary_df = pd.DataFrame(table_data)
print(summary_df)

print("\nTotal:")
print(summary_df.sum(numeric_only=True))

first_ticket_result = analyze_ticket(tickets[0])
print("\nSelected Ticket:")
print("Tokens:", first_ticket_result["tokens"])
print("Punctuation:", first_ticket_result["punct"])
print("Stopwords:", first_ticket_result["stops"])
print("Final Tokens:", first_ticket_result["final"])

In [ ]:
# Q2

pat_long = r'\b[A-Za-z]{6,}\b'
pat_num = r'\b\d+(?:\.\d+){0,2}\b'
pat_cap = r'\b[A-Z][a-zA-Z]*\b'
pat_vowel = r'\b[aeiouAEIOU][a-zA-Z]*\b'
pat_email = r'[\w.-]+@[\w.-]+\.\w+'
pat_url = r'https?://\S+|www\.\S+'
pat_phone = r'\+?\d[\d\s().-]{8,}\d'

for t in tickets:
    long_words = re.findall(pat_long, t)
    nums = re.findall(pat_num, t)
    cap_words = re.findall(pat_cap, t)
    vowel_start = re.findall(pat_vowel, t)

    masked = re.sub(pat_email, '<EMAIL>', t)
    masked = re.sub(pat_url, '<URL>', masked)
    masked = re.sub(pat_phone, '<PHONE>', masked)

    print(f"Long words: {long_words}")
    print(f"Numbers: {nums}")
    print(f"Capitalized words: {cap_words}")
    print(f"Vowel words: {vowel_start}")
    print(f"Replaced: {masked}")
    print()

In [ ]:
# Q3

custom_pattern = (
    r"https?://\S+"
    r"|[\w.-]+@[\w.-]+\.\w+"
    r"|\d+(?:\.\d+){1,2}"
    r"|[A-Za-z]+(?:-[A-Za-z]+)+"
    r"|[A-Za-z]+(?:'[A-Za-z]+)?"
    r"|[.,!?;:]+"
)
custom_tok = RegexpTokenizer(custom_pattern)

def tokenize_custom(sentence):
    return custom_tok.tokenize(sentence)

for sample in TEST_LINES:
    print("Original:", sample)
    print("My tokenizer:", tokenize_custom(sample))
    print("word_tokenize:", word_tokenize(sample))
    print()

for t in tickets:
    print(tokenize_custom(t))

print("\nDifferences:")
print("1. isn't is kept together.")
print("2. Wi-Fi is kept together.")
print("3. 95.5 is kept together.")
print("4. 2.0.1 is kept together.")

In [ ]:
# Q4

ps = PorterStemmer()
ls = LancasterStemmer()
rs = RegexpStemmer("ing$")
wnl = WordNetLemmatizer()

unique_words = set()
for t in tickets:
    unique_words.update(analyze_ticket(t)["final"])

tagged_words = pos_tag(list(unique_words))

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith("V"):
        return wordnet.VERB
    elif treebank_tag.startswith("N"):
        return wordnet.NOUN
    elif treebank_tag.startswith("J"):
        return wordnet.ADJ
    return wordnet.ADV

for w, tag in tagged_words:
    porter_out = ps.stem(w)
    lancaster_out = ls.stem(w)
    regexp_out = rs.stem(w)
    lemma_out = wnl.lemmatize(w, get_wordnet_pos(tag))

    outputs = {porter_out, lancaster_out, regexp_out, lemma_out}
    if len(outputs) > 1:
        print(w, porter_out, lancaster_out, regexp_out, lemma_out)

print("\nRegexp suffix: ing")
print("Reason: -ing words occur in the dataset.")

In [ ]:
# Q5

S1 = [tok for t in tickets for tok in word_tokenize(t)]
S2 = [tok for tok in S1 if tok.isalnum()]
S3 = [tok for t in tickets for tok in analyze_ticket(t)["final"]]
S4 = [ps.stem(tok) for tok in S3]

for label, token_list in [("S1", S1), ("S2", S2), ("S3", S3), ("S4", S4)]:
    total = len(token_list)
    unique = len(set(token_list))
    ttr = unique / total
    print(f"{label}")
    print(f"Total tokens: {total}")
    print(f"Unique tokens: {unique}")
    print(f"TTR: {ttr}")
    print()

print("Top 10 S1:")
print(Counter(S1).most_common(10))
print("\nTop 10 S3:")
print(Counter(S3).most_common(10))

freq_vals = sorted(Counter(S3).values(), reverse=True)
ranks = list(range(1, len(freq_vals) + 1))
plt.loglog(ranks, freq_vals)
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("S3 Log Rank vs Log Frequency")
plt.show()

print("Top 5 S1 bigrams:")
print(Counter(bigrams(S1)).most_common(5))
print("\nTop 5 S3 bigrams:")
print(Counter(bigrams(S3)).most_common(5))

In [ ]:
# Q6

print("(a) TTR changes because stemming combines different forms of the")
print("same word, reducing the number of unique words.")
print()
print("(b) word_tokenize() may split isn't into is and n't, while our")
print("tokenizer keeps isn't together.")
print()
print("(c) university may become univers after stemming. This is undesirable")
print("because univers is not a normal English word.")
print()
print("(d) running can become run using POS-aware lemmatization when it is")
print("identified as a verb. This preserves meaning better than stemming.")